# dSprites Continuous Action — Results

Tables only. Every number is read from `../dSprites/dr_mmd_flow/results/*.csv` and aggregated
in the house order: average within a (data seed, repeat), then within a data seed, then mean
± sd across the three seeds. The pixel, latent and position columns carry a `_mean` suffix
because this design averages each metric over `n_ref` fresh reference draws per run.

**Design.** 64×64 sprite outcome, continuous action, one target policy, one sampler (Riesz
kernel, AdamW with warm-up). Default cell: `n_obs = 16000`, `M = 2000`, no-split nuisance,
no-split autoencoder, observational initialisation, doubly robust target embedding. Every
row has `target_mass = "rescale"`: the target embedding is projected onto total mass 1
before the flow.

**How to read a ratio.** Every flow row is divided by the *count-matched* oracle floor — a
second oracle draw subsampled to the particle count. Raw values are not comparable across
rows with different `M`, because the floor itself moves with `M`. A ratio below 1 means the
output is inside the sampling noise of an independent draw of the same size, not that it
beats the oracle.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import analysis_tables as at

ALL = at.load("dSprites", estimator=None)
DR = ALL[ALL.estimator == "dr"]
BASE = DR[DR.init == "observed"]
DEFAULT = ALL[(ALL.dr_fit == "nosplit") & (ALL.init == "observed")
              & (ALL.n_obs == 16000) & (ALL.n_particles_requested == 2000)]
SEEDS = sorted(int(s) for s in ALL.data_seed.unique())
PIXEL = ["sw2", "sw1", "energy"]
print(f"{len(ALL)} rows   stages {sorted(ALL.tag.unique())}")
print(f"seeds {SEEDS}   estimators {sorted(ALL.estimator.unique())}   "
      f"n_obs {sorted(ALL.n_obs.unique())}")
print(f"M {sorted(ALL.n_particles_requested.unique())}   "
      f"dr_fit {sorted(ALL.dr_fit.unique())}   init {sorted(ALL.init.unique())}")
print(f"target_mass {sorted(ALL.target_mass.unique())}   "
      f"ae_split {sorted(ALL.ae_split.unique())}   "
      f"n_ref {sorted(ALL.n_ref.unique())}")
print(f"default cell: {len(DEFAULT)} rows = {len(SEEDS)} seeds x 3 estimators")

117 rows   stages ['est_crossfit', 'est_headline', 'est_init', 'est_nobs', 'est_particles']
seeds [0, 1, 2]   estimators ['dr', 'ipw', 'plugin']   n_obs [np.int64(2000), np.int64(4000), np.int64(8000), np.int64(16000)]
M [np.int64(250), np.int64(500), np.int64(1000), np.int64(2000)]   dr_fit ['crossfit', 'nosplit']   init ['moment', 'normal', 'observed']
target_mass ['rescale']   ae_split ['none']   n_ref [np.int64(10)]
default cell: 9 rows = 3 seeds x 3 estimators


## 1. Headline (pixels)

In [2]:
g = DEFAULT[DEFAULT.estimator == "dr"]
rows = [(label, [at.summarise(g, f"{m}_{k}_mean") for m in PIXEL])
        for label, k in [("No flow (logged)", "logged"),
                         ("AE round-trip (logged)", "roundtrip"),
                         ("IMMD-Flow (ours)", "flow"),
                         ("Oracle floor (matched)", "floor_noisy_matched"),
                         ("Oracle floor (full)", "floor_noisy")]]
print(at.plain_table(rows, ["SW2", "SW1", "Energy"], first_column="Row", label_width=24,
                     title=f"dSprites   (n_obs = 16000, M = 2000, seeds {SEEDS})"))
ratios = [(m, at.summarise(g, f"{m}_flow_mean")[0]
              / at.summarise(g, f"{m}_floor_noisy_matched_mean")[0]) for m in PIXEL]
print("\nflow / matched floor:   "
      + "   ".join(f"{m.upper()} {r:.2f}x" for m, r in ratios))

dSprites   (n_obs = 16000, M = 2000, seeds [0, 1, 2])

Row                     SW2 ↓                SW1 ↓                Energy ↓
--------------------------------------------------------------------------
No flow (logged)        0.05238 ± 0.00058    0.04063 ± 0.00045    0.60341 ± 0.01331
AE round-trip (logged)  0.05363 ± 0.00059    0.04151 ± 0.00049    0.63222 ± 0.01413
IMMD-Flow (ours)        0.01250 ± 0.00008    0.00917 ± 0.00006    0.01745 ± 0.00032
Oracle floor (matched)  0.00963 ± 0.00008    0.00714 ± 0.00008    0.01634 ± 0.00037
Oracle floor (full)     0.00347 ± 0.00004    0.00256 ± 0.00003    0.00208 ± 0.00007

flow / matched floor:   SW2 1.30x   SW1 1.28x   ENERGY 1.07x


The flow closes most of the pixel gap: SW2 0.05238 → 0.01250 against a matched floor of
0.00963, i.e. **1.30×** the floor, and Energy 0.01745 against 0.01634 (1.07×). The
round-trip row (0.05363) sits beside the logged row, so the autoencoder adds almost nothing
to the pixel distance. The full-size SW2 floor is 0.00347, well under the matched 0.00963,
which is why only the matched ratio is quoted.

## 2. Latent SW2

In [3]:
rows = [(label, [at.summarise(g, f"latent_sw2_{k}_mean")])
        for label, k in [("Initial particles", "initial"),
                         ("No flow (logged)", "logged"),
                         ("IMMD-Flow (ours)", "flow"),
                         ("Oracle floor (matched)", "floor_matched"),
                         ("Oracle floor (full)", "floor")]]
print(at.plain_table(rows, ["Latent SW2"], first_column="Row", label_width=24,
                     title="dSprites   standardized 32-d AE latent   "
                           "(n_obs = 16000, M = 2000)"))
flow = at.summarise(g, "latent_sw2_flow_mean")[0]
print(f"\nflow / matched floor:   "
      f"{flow / at.summarise(g, 'latent_sw2_floor_matched_mean')[0]:.2f}x")

dSprites   standardized 32-d AE latent   (n_obs = 16000, M = 2000)

Row                     Latent SW2 ↓
------------------------------------
Initial particles       0.23854 ± 0.01920
No flow (logged)        0.23961 ± 0.01660
IMMD-Flow (ours)        0.04679 ± 0.00174
Oracle floor (matched)  0.05800 ± 0.00143
Oracle floor (full)     0.02167 ± 0.00076

flow / matched floor:   0.81x


In this design's own standardized 32-d latent the flow reaches 0.04679 against a
count-matched floor of 0.05800 — **0.81×**, inside the sampling noise of a same-size oracle
draw. The initial particles (0.23854) and the logged images (0.23961) are a factor of five
further out. Quote this against the 0.05800 floor beside it, not against another design's
latent SW2.

## 3. FID

In [4]:
rows = [(label, [at.summarise(g, f"fid_{s}_{k}") for s in ("legacy", "clean")])
        for label, k in [("Initial particles", "initial"),
                         ("IMMD-Flow (ours)", "flow"),
                         ("AE round-trip (logged)", "roundtrip"),
                         ("Oracle floor", "floor")]]
print(at.plain_table(rows, ["FID (legacy)", "FID (clean)"], first_column="Row",
                     label_width=24, digits=2,
                     title="dSprites   (n_obs = 16000, M = 2000)",
                     note="Channel-replicated greyscale; a within-design number only."))

dSprites   (n_obs = 16000, M = 2000)

Row                     FID (legacy) ↓       FID (clean) ↓
----------------------------------------------------------
Initial particles       141.55 ± 2.44        86.65 ± 6.46
IMMD-Flow (ours)        136.68 ± 3.11        84.29 ± 7.05
AE round-trip (logged)  140.75 ± 3.39        86.25 ± 6.70
Oracle floor            0.14 ± 0.00          0.14 ± 0.00

Channel-replicated greyscale; a within-design number only.


FID here is computed on channel-replicated greyscale. The autoencoder round-trip of the
logged images scores 140.75 legacy against a floor of 0.14, essentially the flow's own
136.68, so this column measures the decoder and not the flow. Clean-FID gives smaller
absolute values (84.29 flow, 86.25 round-trip) against the same 0.14 floor and the same
ordering. A within-design sanity check only.

## 4. Target-embedding estimator

In [5]:
print(at.estimator_table(DEFAULT, "dSprites"))
print()
print(at.plain_table(
    [(at.ESTIMATOR_LABELS[e],
      [at.summarise(DEFAULT[DEFAULT.estimator == e], c)
       for c in ("sum_v_raw", "sum_v", "v_ess", "v_max_abs", "mean_transport")])
     for e in at.ESTIMATORS],
    ["mass returned", "mass used", "v ESS", "max |v|", "mean transport"],
    first_column="Target embedding", label_width=26, arrow=False,
    title="Weight vector and transport   (n_obs = 16000, M = 2000)"))

dSprites   target embedding estimator

Target embedding          SW2 ↓                FID ↓                Latent SW2 ↓         SW1 ↓                Energy ↓
----------------------------------------------------------------------------------------------------------------------
Plug-in (CME only)        0.01250 ± 0.00006    136.721 ± 3.174      0.04681 ± 0.00177    0.00918 ± 0.00005    0.01750 ± 0.00030
IPW (ratios only)         0.01461 ± 0.00024    138.575 ± 3.643      0.06790 ± 0.00744    0.01087 ± 0.00018    0.03221 ± 0.00218
Doubly robust (default)   0.01250 ± 0.00008    136.682 ± 3.114      0.04679 ± 0.00174    0.00917 ± 0.00006    0.01745 ± 0.00032

One logged sample, one nuisance solve, one set of initial particles and one flow budget per run;
only the target weight vector differs.

Weight vector and transport   (n_obs = 16000, M = 2000)

Target embedding          mass returned        mass used            v ESS                max |v|              mean transport
-------------------

Plug-in and DR are indistinguishable on every column; IPW is worse — pixel SW2 0.01461
against 0.01250, latent SW2 0.06790 against 0.04679, Energy 0.03221 against 0.01745. FID
spans under 2 points across all three, as the previous section predicts. The masses returned
differ (0.99860 / 0.91389 / 0.99868) and are all projected to `sum_v = 1`; mean transport is
4.350 / 4.130 / 4.356, so IPW does not move the particles further than DR.

## 5. Ablations

One helper, four sweeps. Each row shows pixel and latent SW2 against the floor at that row's
own particle count.

In [6]:
def sweep(frame, by, order, labels, title):
    print(title)
    print(f"  {'':18s} {'pixel':>17s}   {'latent':>17s}")
    print(f"  {'':18s} {'SW2':>9s} {'ratio':>7s}   {'SW2':>9s} {'ratio':>7s}")
    for value in order:
        h = frame[frame[by] == value]
        out = []
        for flow, floor in [("sw2_flow_mean", "sw2_floor_noisy_matched_mean"),
                            ("latent_sw2_flow_mean", "latent_sw2_floor_matched_mean")]:
            m, _ = at.summarise(h, flow)
            fo, _ = at.summarise(h, floor)
            out.append((m, m / fo))
        print(f"  {labels[value]:18s} {out[0][0]:9.5f} {out[0][1]:6.2f}x   "
              f"{out[1][0]:9.5f} {out[1][1]:6.2f}x")
    print()


NOSPLIT = BASE[BASE.dr_fit == "nosplit"]
sweep(NOSPLIT[NOSPLIT.n_particles_requested == 2000], "n_obs", [2000, 4000, 8000, 16000],
      {n: f"n_obs = {n}" for n in (2000, 4000, 8000, 16000)},
      "Observations   (M = 2000, no-split)")
sweep(NOSPLIT[NOSPLIT.n_obs == 16000], "n_particles_requested", [250, 500, 1000, 2000],
      {m: f"M = {m}" for m in (250, 500, 1000, 2000)},
      "Particles   (n_obs = 16000, no-split)")
sweep(BASE[(BASE.n_obs == 16000) & (BASE.n_particles_requested == 2000)], "dr_fit",
      ["nosplit", "crossfit"], {"nosplit": "No-split", "crossfit": "Cross-fitted"},
      "Nuisance fitting   (n_obs = 16000, M = 2000)")
sweep(DR[(DR.dr_fit == "nosplit") & (DR.n_obs == 16000)
         & (DR.n_particles_requested == 2000)], "init", ["observed", "moment", "normal"],
      {"observed": "Observed (P_Y)", "moment": "Moment-matched",
       "normal": "Standard normal"},
      "Initialisation   (n_obs = 16000, M = 2000)")

Observations   (M = 2000, no-split)
                                 pixel              latent
                           SW2   ratio         SW2   ratio
  n_obs = 2000         0.01902   1.99x     0.09309   1.66x
  n_obs = 4000         0.01460   1.51x     0.06373   1.22x
  n_obs = 8000         0.01295   1.32x     0.05151   0.94x
  n_obs = 16000        0.01250   1.30x     0.04679   0.81x



Particles   (n_obs = 16000, no-split)
                                 pixel              latent
                           SW2   ratio         SW2   ratio
  M = 250              0.02281   0.83x     0.11071   0.71x


  M = 500              0.01756   0.95x     0.08224   0.73x
  M = 1000             0.01421   1.08x     0.06002   0.72x


  M = 2000             0.01250   1.30x     0.04679   0.81x

Nuisance fitting   (n_obs = 16000, M = 2000)
                                 pixel              latent
                           SW2   ratio         SW2   ratio
  No-split             0.01250   1.30x     0.04679   0.81x


  Cross-fitted         0.01244   1.29x     0.04549   0.78x

Initialisation   (n_obs = 16000, M = 2000)
                                 pixel              latent
                           SW2   ratio         SW2   ratio
  Observed (P_Y)       0.01250   1.30x     0.04679   0.81x
  Moment-matched       0.01236   1.28x     0.04681   0.81x
  Standard normal      0.01238   1.29x     0.04656   0.81x



**Observations.** Monotone over an 8× increase in data: 1.99× → 1.30× pixel, 1.66× → 0.81×
latent.

**Particles.** The raw pixel column falls 0.02281 → 0.01250 while the ratio *rises* 0.83× →
1.30×, because the matched floor falls faster than the flow does; the latent ratio stays
between 0.71× and 0.81×. Read the ratio, not the raw value.

**Nuisance fitting.** No difference: 1.30× vs 1.29× pixel, 0.81× vs 0.78× latent.

**Initialisation.** No difference either — 1.30× / 1.28× / 1.29× pixel, 0.81× on all three
latent rows.

## 6. Appendix: the snapped position law

Each output image is snapped to the nearest sprite and scored on its (x, y) pixel position,
a 2-D summary of the same samples.

In [7]:
rows = [(label, [at.summarise(g, f"pos_{m}_{k}_mean") for m in PIXEL])
        for label, k in [("No flow (logged)", "logged"),
                         ("IMMD-Flow (ours)", "flow"),
                         ("Oracle floor (matched)", "floor_matched"),
                         ("Oracle floor (full)", "floor")]]
print(at.plain_table(rows, ["SW2", "SW1", "Energy"], first_column="Row", label_width=24,
                     title="dSprites   snapped sprite position   (n_obs = 16000, M = 2000)"))
pos = at.summarise(g, "pos_sw2_flow_mean")[0]
print(f"\nflow / matched floor:   "
      f"SW2 {pos / at.summarise(g, 'pos_sw2_floor_matched_mean')[0]:.2f}x")
for label, key in [("flow", "flow"), ("oracle", "oracle"), ("logged", "logged")]:
    x = at.summarise(g, f"pos_mean_{key}_x")[0]
    y = at.summarise(g, f"pos_mean_{key}_y")[0]
    print(f"mean position ({label:6s}):   x {x:6.2f}   y {y:6.2f}")

dSprites   snapped sprite position   (n_obs = 16000, M = 2000)

Row                     SW2 ↓                SW1 ↓                Energy ↓
--------------------------------------------------------------------------
No flow (logged)        6.331 ± 0.056        5.456 ± 0.043        4.023 ± 0.072
IMMD-Flow (ours)        0.29933 ± 0.01889    0.22931 ± 0.01469    0.00220 ± 0.00015
Oracle floor (matched)  0.49013 ± 0.05486    0.39536 ± 0.04538    0.02372 ± 0.00570
Oracle floor (full)     0.14614 ± 0.00309    0.10370 ± 0.00287    0.00161 ± 0.00002

flow / matched floor:   SW2 0.61x
mean position (flow  ):   x  18.50   y  18.50
mean position (oracle):   x  18.50   y  18.49
mean position (logged):   x  12.39   y  12.49


Flow SW2 is 0.29933 against a matched floor of 0.49013 — **0.61×**, again inside the floor —
from a logged value of 6.331. The flow's mean position (18.50, 18.50) matches the oracle's
(18.50, 18.49), against a logged mean of (12.39, 12.49).

## 7. Coverage

In [8]:
print(at.plain_table(
    [(tag, [(len(h), 0.0), (len(h) / 3, 0.0)]) for tag, h in ALL.groupby("tag", observed=True)],
    ["rows", "cells"], first_column="Launcher stage", label_width=18, digits=0, arrow=False,
    title="dSprites   what is on disk",
    note="cells = rows / 3 estimators; 3 seeds each."))
print(f"\ntotal {len(ALL)} rows, all five stages complete at {len(SEEDS)} seeds.")

dSprites   what is on disk

Launcher stage    rows                 cells
--------------------------------------------
est_crossfit      36 ± 0               12 ± 0
est_headline      9 ± 0                3 ± 0
est_init          18 ± 0               6 ± 0
est_nobs          27 ± 0               9 ± 0
est_particles     27 ± 0               9 ± 0

cells = rows / 3 estimators; 3 seeds each.

total 117 rows, all five stages complete at 3 seeds.
